# R12 · Block distance vs whole-matrix distance (supplementary Fig S9)

How much does leaving out the between-block entries change the distance? Breast, seed-0 layout: in each of the three
largest niches, 1,000 random distinct pairs of index tiles (one generator, `default_rng(0)`, drawn niche after niche),
and for each pair the whole-matrix log-Euclidean distance d_W = ‖log C_i − log C_j‖_F / √p and the block distance
d_B (Spindle-naive's scan, over the niche's blocks). Both from vectors already in the cache, on the same scale; a few
pairs are recomputed from the covariances as a check (`block_distance.py`).

* **Prerequisites:** `R0_index_cache` (staged covariances + whole-matrix vectors, seed-0 Spindle-naive index).
Per dataset (supplementary Fig S10): every dataset, 1,000 random distinct pairs of index tiles drawn uniformly from all
pairs of tiles in the same niche (d_B exists only within a niche), `default_rng(0)` per dataset.

* **Writes:** `results/block_distance/pairs.csv` (one row per tile pair), `niches.csv` (tiles and blocks of the
  plotted niches); `dataset_pairs.csv`, `datasets.csv` (per dataset).
* **Runtime / memory:** a few minutes; < 8 GB (whole-matrix vectors memory-mapped).
  `sbatch experiments/common/run_nb.sbatch experiments/compute/R12_block_distance.ipynb`

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

import block_distance as bd
import data
import results_io as rio

TOPIC = "block_distance"
KEY = "breast_cancer"
FORCE = False
OUT = rio.topic(TOPIC)
print(data.stamp())

{'tag': 'method-v2', 'describe': 'method-v3-4-gb026ef3-dirty', 'commit': 'b026ef3'}


## Pairs and distances

In [2]:
if FORCE or not (OUT / "pairs.csv").exists():
    pairs, niches = bd.pairs(KEY)
    bd.check(KEY, pairs)
    tag = data.stamp()["tag"]
    rio.write_csv(pairs.assign(key=KEY, tag=tag), TOPIC, "pairs.csv",
                  f"{KEY}, seed-0 layout, the {bd.N_NICHES} largest niches: {bd.N_PAIRS} random distinct pairs of index "
                  "tiles per niche (default_rng(0), niche after niche); whole-matrix log-Euclidean distance d_whole and "
                  "block distance d_block (both / sqrt(p), shrunk covariances); tiles in prepare() order")
    rio.write_csv(niches.assign(key=KEY, tag=tag), TOPIC, "niches.csv",
                  "the plotted niches, largest first: index tiles, blocks, genes p")
pairs = pd.read_csv(OUT / "pairs.csv")
niches = pd.read_csv(OUT / "niches.csv")
niches

,niche,tiles,blocks,p,key,tag
0,0,664,16,313,breast_cancer,method-v2
1,1,493,15,313,breast_cancer,method-v2
2,2,411,17,313,breast_cancer,method-v2


## Agreement per niche (shown, not saved)

In [3]:
def summary(g):
    gap = g.d_whole - g.d_block
    return pd.Series({"pairs": len(g),
                      "pearson_r": pearsonr(g.d_whole, g.d_block)[0],
                      "spearman_rho": spearmanr(g.d_whole, g.d_block)[0],
                      "median_gap": float(np.median(gap)),
                      "median_ratio_block_whole": float(np.median(g.d_block / g.d_whole)),
                      "frac_block_le_whole": float(np.mean(g.d_block <= g.d_whole))})

pairs.groupby("niche", sort=False).apply(summary).round(3)

,pairs,pearson_r,spearman_rho,median_gap,median_ratio_block_whole,frac_block_le_whole
niche,,,,,,
0,1000.0,0.969,0.950,0.255,0.783,0.997
1,1000.0,0.978,0.942,0.300,0.803,0.997
2,1000.0,0.985,0.955,0.195,0.842,0.980


## Per dataset: pairs within any niche (Fig S10)

In [4]:
if FORCE or not (OUT / "dataset_pairs.csv").exists():
    tag = data.stamp()["tag"]
    dp, di = [], []
    for key in data.indexed_keys():
        pr, info = bd.dataset_pairs(key)
        bd.check(key, pr, per_niche=1)  # one pair per niche, recomputed from the covariances
        dp.append(pr.assign(key=key))
        di.append({"key": key, **info, "niches_sampled": pr.niche.nunique()})
        print(f"{key}: {info}", flush=True)
    rio.write_csv(pd.concat(dp, ignore_index=True).assign(tag=tag), TOPIC, "dataset_pairs.csv",
                  f"every dataset, seed-0 layout: {bd.N_PAIRS} random distinct pairs of index tiles drawn uniformly from "
                  "all pairs of tiles in the same niche (default_rng(0) per dataset); niche, d_whole, d_block as in pairs.csv")
    rio.write_csv(pd.DataFrame(di).assign(tag=tag), TOPIC, "datasets.csv",
                  "per dataset: index tiles, niches, genes p, niches with at least one sampled pair")
dpairs = pd.read_csv(OUT / "dataset_pairs.csv")
dinfo = pd.read_csv(OUT / "datasets.csv")
dpairs.groupby("key", sort=False).apply(summary).round(3).join(dinfo.set_index("key")[["tiles", "niches", "p"]])

niche 0: d_W 1.63658 vs cached 1.63658 | d_B 1.25393 vs cached 1.25393
niche 1: d_W 1.09001 vs cached 1.09001 | d_B 0.82631 vs cached 0.82631
niche 2: d_W 1.54212 vs cached 1.54212 | d_B 1.10027 vs cached 1.10027
niche 3: d_W 1.95880 vs cached 1.95880 | d_B 1.43917 vs cached 1.43917
skin_melanoma: {'tiles': 839, 'niches': 4, 'p': 382}
niche 0: d_W 2.64787 vs cached 2.64787 | d_B 2.11708 vs cached 2.11708
niche 1: d_W 0.77255 vs cached 0.77255 | d_B 0.60337 vs cached 0.60337
kidney_nondiseased: {'tiles': 705, 'niches': 2, 'p': 377}
niche 0: d_W 1.08313 vs cached 1.08313 | d_B 0.92932 vs cached 0.92932
niche 1: d_W 1.50209 vs cached 1.50209 | d_B 1.28017 vs cached 1.28017
niche 2: d_W 1.22737 vs cached 1.22737 | d_B 0.95619 vs cached 0.95619
niche 3: d_W 1.72107 vs cached 1.72107 | d_B 1.49887 vs cached 1.49887
breast_cancer: {'tiles': 1840, 'niches': 4, 'p': 313}
niche 0: d_W 1.29465 vs cached 1.29465 | d_B 0.98253 vs cached 0.98253
niche 1: d_W 1.29598 vs cached 1.29598 | d_B 1.05034 v

,pairs,pearson_r,spearman_rho,median_gap,median_ratio_block_whole,frac_block_le_whole,tiles,niches,p
key,,,,,,,,,
skin_melanoma,1000.0,0.951,0.899,0.372,0.745,1.000,839,4,382
kidney_nondiseased,1000.0,0.982,0.982,0.167,0.836,1.000,705,2,377
breast_cancer,1000.0,0.974,0.935,0.255,0.797,0.994,1840,4,313
lung_cancer,1000.0,0.953,0.912,0.306,0.770,0.995,1535,3,377
pancreatic_cancer,1000.0,0.926,0.890,0.420,0.664,1.000,2061,4,474
lymph_node,1000.0,0.898,0.882,0.372,0.699,0.999,2354,5,377
lymph_node_5k,1000.0,0.509,0.192,0.874,0.519,1.000,6972,15,400
brain_cancer,1000.0,0.882,0.795,0.333,0.674,1.000,9558,18,480


In [5]:
rio.write_manifest(TOPIC, "R12_block_distance.ipynb", inputs=["results/_cache/<key>/whole.npy (R0)",
                                                              "results/_cache/<key>/covs.npy (R0)",
                                                              "results/_cache/<key>/seed0/naive.pkl (R0)"])

wrote results/block_distance/manifest.json: 4 files; peak memory 1.65 GB


{'topic': 'block_distance',
 'notebook': 'R12_block_distance.ipynb',
 'date': '2026-11-11 20:00',
 'tag': 'method-v2',
 'describe': 'method-v3-4-gb026ef3-dirty',
 'commit': 'b026ef3',
 'inputs': ['results/_cache/<key>/whole.npy (R0)',
  'results/_cache/<key>/covs.npy (R0)',
  'results/_cache/<key>/seed0/naive.pkl (R0)'],
 'peak_rss_gb': 1.65,
 'notes': '',
 'files': {'dataset_pairs.csv': {'description': 'every dataset, seed-0 layout: 1000 random distinct pairs of index tiles drawn uniformly from all pairs of tiles in the same niche (default_rng(0) per dataset); niche, d_whole, d_block as in pairs.csv',
   'rows': 8000,
   'bytes': 583058},
  'datasets.csv': {'description': 'per dataset: index tiles, niches, genes p, niches with at least one sampled pair',
   'rows': 8,
   'bytes': 339},
  'niches.csv': {'description': 'the plotted niches, largest first: index tiles, blocks, genes p',
   'rows': 3,
   'bytes': 140},
  'pairs.csv': {'description': 'breast_cancer, seed-0 layout, the 3 lar